In [0]:
# TODO remove any unused 
from pyspark.sql.functions import current_timestamp, col, flatten, array, expr, lit
from datetime import datetime, date, timezone
from time import time
import urllib.request
import os
import shutil
import json
from delta.tables import DeltaTable
from utils.data_models import make_1h_price_df

In [ ]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# Get the ingest_timestamp file path for the lastest json file
# used to get only the most recent data
ingest_timestamp = dbutils.jobs.taskValues.get(
    taskKey = "00_ingest_1h_prices",
    key="ingest_timestamp",
    debugValue="1770405983")
file_path = f"/Volumes/{catalog}/00_landing/data_sources/1h_prices/1h_prices_{ingest_timestamp}.json"

In [0]:
# open the file so we can transform it and save as parquet file
with open(file_path, 'r') as file:
    data = json.load(file)

In [0]:
# Convert the data to a Spark DataFrame
df_1h_prices = make_df_1h_price(spark, data)

In [0]:
# Insert df_1h_prices into 'runescape.01_bronze.1h_prices' table

targetDF = DeltaTable.forName(spark, f"{catalog}.01_bronze.1h_prices")
dfUpdates = df_1h_prices

targetDF.alias("t") .\
  merge(
    source = dfUpdates.alias("s"),
    condition = "t.time = s.time") .\
  whenNotMatchedInsertAll() .\
  execute()

In [0]:
%skip
# Write the DataFrame to a Unity Catalog managed Delta table in the bronze schema, appending the new data
df_1h_prices.write.mode("append").saveAsTable("runescape.01_bronze.1h_prices")